# Cayley/RoBERTa Colab Runtime Setup

This notebook runs on a Colab GPU kernel while loading your project files from Drive, GitHub, or an uploaded zip.

## 1. Runtime check

In Colab, use `Runtime > Change runtime type > GPU` before running the rest of the notebook.

In [8]:
import os
import platform
import sys

print('Python:', sys.version)
print('Platform:', platform.platform())
!nvidia-smi || true

Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
Mon Aug 31 06:52:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P0             28W /   70W |     661MiB /  15360MiB |      0%      Default |
|                                 

## 2. Choose project source

Use `drive` if your project folder is in Google Drive. Use `github` if the repo is pushed. Use `upload` for a zipped copy of the project.

In [9]:
PROJECT_SOURCE = 'github'  # 'drive', 'github', or 'upload'

# Drive settings
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/cayley'

# GitHub settings. Leave GITHUB_TOKEN empty for public repos.
# For private repos, add a Colab Secret named GITHUB_TOKEN with repo read access.
GITHUB_REPO = 'https://github.com/picramide/cayley.git'
GITHUB_BRANCH = 'main'
GITHUB_TOKEN = ''
GITHUB_TOKEN_SECRET = 'GITHUB_TOKEN'

RUNTIME_PROJECT_DIR = '/content/cayley'
REQUIREMENTS_FILE = 'requirements-colab.txt'

## 3. Make project files available to Colab

In [10]:
from pathlib import Path
import os
import shutil
import subprocess


def run(cmd, cwd=None):
    print('+', cmd, flush=True)
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    completed = subprocess.run(cmd, shell=True, check=False, cwd=cwd, env=env)
    if completed.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {completed.returncode}: {cmd}')
    return completed

if PROJECT_SOURCE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    project_dir = Path(DRIVE_PROJECT_DIR)
    if not project_dir.exists():
        raise FileNotFoundError(f'Drive project folder not found: {project_dir}')

elif PROJECT_SOURCE == 'github':
    repo_url = GITHUB_REPO
    github_token = GITHUB_TOKEN
    if not github_token:
        try:
            from google.colab import userdata
            github_token = userdata.get(GITHUB_TOKEN_SECRET) or ''
        except Exception:
            github_token = ''
    if github_token:
        repo_url = repo_url.replace('https://', f'https://x-access-token:{github_token}@')
    elif 'github.com' in repo_url:
        print('No GitHub token found. Public repos can clone without one; private repos need a Colab Secret named GITHUB_TOKEN.')
    project_dir = Path(RUNTIME_PROJECT_DIR)
    if project_dir.exists():
        shutil.rmtree(project_dir)
    run(f'git clone --branch {GITHUB_BRANCH} --depth 1 {repo_url} {project_dir}')

elif PROJECT_SOURCE == 'upload':
    from google.colab import files
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.endswith('.zip')]
    if not zip_names:
        raise ValueError('Upload a .zip file containing the project.')
    project_dir = Path(RUNTIME_PROJECT_DIR)
    if project_dir.exists():
        shutil.rmtree(project_dir)
    project_dir.mkdir(parents=True)
    run(f'unzip -q {zip_names[0]} -d {project_dir}')
    children = [p for p in project_dir.iterdir() if p.is_dir()]
    if len(children) == 1 and not (project_dir / REQUIREMENTS_FILE).exists():
        project_dir = children[0]

else:
    raise ValueError(f'Unknown PROJECT_SOURCE: {PROJECT_SOURCE}')

print('Project dir:', project_dir)
print('Top-level files:', sorted(p.name for p in project_dir.iterdir())[:30])


No GitHub token found. Public repos can clone without one; private repos need a Colab Secret named GITHUB_TOKEN.
+ git clone --branch main --depth 1 https://github.com/picramide/cayley.git /content/cayley
Project dir: /content/cayley
Top-level files: ['.git', '.gitignore', 'BENCHMARKING_NOTES.md', 'COLAB.md', 'README.md', 'cayley', 'colab_roberta_cayley_setup.ipynb', 'pyproject.toml', 'requirements-colab.txt', 'scripts']


## 4. Install dependencies and project

In [11]:
import sys
from pathlib import Path

req_path = Path(project_dir) / REQUIREMENTS_FILE
if req_path.exists():
    run(f'{sys.executable} -m pip install -q -r {req_path}')
else:
    run(f'{sys.executable} -m pip install -q torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121')
    run(f'{sys.executable} -m pip install -q transformers datasets evaluate accelerate scikit-learn networkx pandas tqdm matplotlib seaborn einops wandb')

if (Path(project_dir) / 'pyproject.toml').exists() or (Path(project_dir) / 'setup.py').exists():
    run(f'{sys.executable} -m pip install -q -e {project_dir}')

if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

print('sys.path[0]:', sys.path[0])

+ /usr/bin/python3 -m pip install -q -r /content/cayley/requirements-colab.txt
+ /usr/bin/python3 -m pip install -q -e /content/cayley
sys.path[0]: /content/cayley


## 5. Verify PyTorch, Transformers, and RoBERTa

In [12]:
import torch
import transformers
from transformers import AutoConfig, AutoModel, AutoTokenizer

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
print('transformers:', transformers.__version__)

model_name = 'roberta-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
config = AutoConfig.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, config=config)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

inputs = tokenizer('cayley graph transformer pattern benchmark smoke test', return_tensors='pt').to(device)
with torch.no_grad():
    outputs = model(**inputs)
print('last_hidden_state:', tuple(outputs.last_hidden_state.shape))

torch: 2.11.0+cu128
cuda available: True
gpu: Tesla T4
transformers: 5.15.1


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


last_hidden_state: (1, 11, 768)


## 6. Run MRPC with bipartite Cayley mask

This runs MRPC using the newly added bipartite Cayley mask. The mask is designed for NLI/entailment tasks where:
- The first token (CLS) attends to all tokens (global hub)
- All tokens attend to CLS
- Local window attention within premise and hypothesis
- Cross-attention between premise and hypothesis tokens

Training uses full MRPC train/validation splits with the same parameters as the existing window benchmark.

In [ ]:
AUTO_DOWNLOAD_RESULTS = True
N_EARLY_DENSE_RESULTS_FILE = 'results/mrpc_nearly_dense.jsonl'
N_EARLY_DENSE_MASK_PATH = 'masks/nearly_dense_128_4.pt'

run(
    'python -u scripts/generate_masks.py '
    '--kind nearly_dense '
    '--seq 128 '
    '--drop_degree 4 '
    f'--output {N_EARLY_DENSE_MASK_PATH}',
    cwd=project_dir,
)

BENCHMARK_BASE = (
    'python -u scripts/benchmark_roberta_glue.py '
    '--task_name mrpc '
    '--dataset_name nyu-mll/glue '
    '--model_name FacebookAI/roberta-base '
    '--do_train '
    '--do_eval '
    '--max_length 128 '
    '--num_train_epochs 5.0 '
    '--seed 42 '
    '--per_device_train_batch_size 8 '
    '--per_device_eval_batch_size 16 '
    '--learning_rate 2e-5 '
)

run(
    BENCHMARK_BASE +
    '--mask_name nearly_dense '
    f'--mask_path {N_EARLY_DENSE_MASK_PATH} '
    '--output_dir outputs/mrpc_nearly_dense '
    f'--results_file {N_EARLY_DENSE_RESULTS_FILE} '
    '--run_name mrpc_nearly_dense ',
    cwd=project_dir,
)

result_path = project_dir / N_EARLY_DENSE_RESULTS_FILE
if not result_path.exists():
    raise FileNotFoundError(f'Expected results file not found: {result_path}')
print(f'Results saved at: {result_path}')
print(result_path.read_text())

if AUTO_DOWNLOAD_RESULTS:
    try:
        from google.colab import files
        files.download(str(result_path))
    except Exception as exc:
        print(f'Automatic browser download failed: {exc}')

+ python -u scripts/generate_masks.py --kind bipartite --seq 128 --premise_len 64 --local_window 3 --cross_window 2 --output masks/bipartite_128_64.pt
+ python -u scripts/benchmark_roberta_glue.py --task_name mrpc --dataset_name nyu-mll/glue --model_name FacebookAI/roberta-base --do_train --do_eval --max_length 128 --num_train_epochs 5.0 --seed 42 --per_device_train_batch_size 8 --per_device_eval_batch_size 16 --learning_rate 2e-5 --mask_name bipartite --mask_path masks/bipartite_128_64.pt --output_dir outputs/mrpc_bipartite --results_file results/mrpc_bipartite.jsonl --run_name mrpc_bipartite
Results saved at: /content/cayley/results/mrpc_bipartite.jsonl
{"dataset_name": "nyu-mll/glue", "learning_rate": 2e-05, "mask_name": "bipartite", "mask_path": "masks/bipartite_128_64.pt", "max_length": 128, "metrics": {"epoch": 5.0, "eval_accuracy": 0.6446078431372549, "eval_f1": 0.7433628318584071, "eval_loss": 0.7943180799484253, "eval_runtime": 0.7023, "eval_samples_per_second": 580.967, "eval

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>